# 04 - Supplementary Tables S3 and S4

In [ ]:
# --- config ---
HIDALGO_RCA_INPUT = "counts"  # "counts" | "binary"
HIDALGO_GRAPH = "plot_graph"  # "plot_graph" | "full_phi" | "disparity_backbone"

MARKET_REF = "Versus"
CATEGORY2_REF = "Cannabis"
COVARIANCE_TYPE = "cluster_product"
SAVE_PREFIX = None  # None = auto from selected network

import os
import sys
import importlib
from pathlib import Path
from typing import Dict, Tuple

import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
import dnm_viz_utils as dvu
dvu = importlib.reload(dvu)

repo_root = dvu.repo_root
load_master_drugs = dvu.load_master_drugs
load_category2_most_common_mapping = dvu.load_category2_most_common_mapping
MARKETS = dvu.MARKETS
MARKET_FILE_NAMES = dvu.MARKET_FILE_NAMES

os.chdir(repo_root())
ROOT = repo_root()
OUT = ROOT / "prescription_reports/new_category2_networks"
TAB = OUT / "tables"
NET = OUT / "networks"
TAB.mkdir(parents=True, exist_ok=True)
NET.mkdir(parents=True, exist_ok=True)

def hidalgo_suffix() -> str:
    return "" if HIDALGO_RCA_INPUT == "counts" else f"_{HIDALGO_RCA_INPUT}"

def network_label() -> str:
    return f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}"

PREFIX = SAVE_PREFIX or f"regression_{network_label()}"
print("Network:", network_label())
print("Saving tables with prefix:", PREFIX)



## Build or load the selected network

The Hidalgo product spaces are loaded from notebook 01 using the settings reported in the manuscript.


In [ ]:
def add_listing_id(df: pd.DataFrame) -> pd.Series:
    website = df["Website"].fillna("").astype(str).str.strip()
    pages = df["Pages"].fillna("").astype(str).str.strip()
    vendor = df["ID_Num"].fillna("").astype(str).str.strip()
    names = df["Productname"].fillna("").astype(str).str.strip().str.casefold()
    cat3 = df["Category 3"].fillna("").astype(str).str.strip()
    fallback = website + " | " + vendor + " | " + names + " | " + cat3
    valid_pages = pages.ne("") & ~pages.str.lower().isin(["nan", "none"])
    out = pages.where(valid_pages, fallback)
    return out.where(~website.isin(["WhiteHouse Market", "ToRReZ"]), fallback).astype(str).str.strip()


def giant_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    return G.subgraph(max(nx.connected_components(G), key=len)).copy()


def load_saved_graphml(path: Path) -> nx.Graph | None:
    if not path.exists():
        return None
    G = nx.read_graphml(path)
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})
    return giant_component(G)








def get_network(market: str, df_drugs: pd.DataFrame) -> nx.Graph:
    tag = MARKET_FILE_NAMES[market]
    path = NET / f"{tag}_hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}.graphml"
    graph = load_saved_graphml(path)
    if graph is None:
        raise FileNotFoundError(f"Missing {path}. Run notebook 01 first.")
    return graph


## Build the product-market regression panel

 `L` is vendor listing count, `cat2_scope_v` is the number of harmonized Category 2 groups represented in a vendor's drug listings, and `product_total_listings` is the product-level total number of distinct listings across all sellers of that Category 3 product. Category 3 breadth (`K`) is retained only as a robustness diagnostic and is not used in the main models.


In [ ]:
def vendor_product_listing_counts(df_market: pd.DataFrame) -> pd.DataFrame:
    d = df_market.dropna(subset=["ID_Num", "Category 3"]).copy()
    d["ID_Num"] = d["ID_Num"].astype(str).str.strip()
    d["Category 3"] = d["Category 3"].astype(str).str.strip()
    d["_listing_id"] = add_listing_id(d)
    d = d[(d["ID_Num"] != "") & (d["Category 3"] != "") & (d["_listing_id"] != "")]
    out = d.groupby(["ID_Num", "Category 3"], sort=False)["_listing_id"].nunique().reset_index(name="x_vp")
    return out[out["x_vp"] > 0]


def vendor_level_supply_stats(df_market: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame:
    d = df_market.dropna(subset=["ID_Num", "Category 3"]).copy()
    d["ID_Num"] = d["ID_Num"].astype(str).str.strip()
    d["Category 3"] = d["Category 3"].astype(str).str.strip()
    d["category2_harmonized"] = d["Category 3"].map(lambda p: prod_to_cat2.get(p, "Unknown"))
    d["_listing_id"] = add_listing_id(d)
    d = d[(d["ID_Num"] != "") & (d["Category 3"] != "") & (d["_listing_id"] != "")]
    vp = d.groupby(["ID_Num", "Category 3"], sort=False)["_listing_id"].nunique().reset_index(name="x_vp")
    L = d.groupby("ID_Num", sort=False)["_listing_id"].nunique().rename("L_v").reset_index()
    K = d.groupby("ID_Num", sort=False)["Category 3"].nunique().rename("K_v").reset_index()
    cat2_scope = d.groupby("ID_Num", sort=False)["category2_harmonized"].nunique().rename("cat2_scope_v").reset_index()
    vendor_total = vp.groupby("ID_Num", sort=False)["x_vp"].sum().rename("vendor_total_listings").reset_index()
    return L.merge(K, on="ID_Num").merge(cat2_scope, on="ID_Num").merge(vendor_total, on="ID_Num")


def product_vendor_supply_stats(df_market: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame:
    vp = vendor_product_listing_counts(df_market)
    vstats = vendor_level_supply_stats(df_market, prod_to_cat2)
    merged = vp.merge(vstats, on="ID_Num", how="left")
    rows = []
    for product, grp in merged.groupby("Category 3", sort=False):
        rows.append({
            "product_cat3": str(product),
            "n_sellers": int(grp["ID_Num"].nunique()),
            "mean_L": float(grp["L_v"].mean()),
            "max_L": float(grp["L_v"].max()),
            "mean_K": float(grp["K_v"].mean()),
            "max_K": float(grp["K_v"].max()),
            "mean_cat2_scope": float(grp["cat2_scope_v"].mean()),
            "max_cat2_scope": float(grp["cat2_scope_v"].max()),
            "product_total_listings": int(grp["x_vp"].sum()),
            "mean_vendor_total_listings": float(grp["vendor_total_listings"].mean()),
            "max_vendor_total_listings": float(grp["vendor_total_listings"].max()),
        })
    return pd.DataFrame(rows)


def category2_distinct_cat3_per_market(df_drugs: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame:
    d = df_drugs.dropna(subset=["Website", "Category 3"]).copy()
    d["product_cat3"] = d["Category 3"].astype(str).str.strip()
    d["category2"] = d["product_cat3"].map(lambda p: prod_to_cat2.get(p, "Unknown"))
    return d.groupby(["Website", "category2"], sort=False)["product_cat3"].nunique().reset_index(name="n_cat3_in_cat2").rename(columns={"Website": "market"})


def count_reviews_by_product(df_market: pd.DataFrame) -> Dict[str, int]:
    return {str(k): int(v) for k, v in df_market["Category 3"].dropna().astype(str).str.strip().value_counts().items()}


def centrality_frame(market: str, G: nx.Graph, prod_to_cat2: dict, reviews_by_product: dict) -> pd.DataFrame:
    degree_centrality = nx.degree_centrality(G) if G.number_of_nodes() > 1 else {n: 0.0 for n in G.nodes()}
    rows = []
    for node in G.nodes():
        node = str(node)
        reviews = int(reviews_by_product.get(node, 0))
        if reviews <= 0:
            continue
        weights = [float(G[node][nbr].get("phi", G[node][nbr].get("weight", 0.0))) for nbr in G[node]]
        category2 = prod_to_cat2.get(node, "Unknown")
        rows.append({
            "network_kind": "hidalgo",
            "network_label": network_label(),
            "market": market,
            "product_cat3": node,
            "category2": category2,
            "is_prescription": int(category2 == "Prescription"),
            "reviews": reviews,
            "log1p_reviews": float(np.log1p(reviews)),
            "degree_centrality": float(degree_centrality.get(node, 0.0)),
            "weighted_degree": float(np.sum(weights)),
        })
    return pd.DataFrame(rows)


def build_panel() -> pd.DataFrame:
    df_drugs = load_master_drugs()
    df_drugs = df_drugs[df_drugs["Website"].isin(MARKETS)].copy()
    prod_to_cat2 = load_category2_most_common_mapping(df_drugs)

    frames = []
    for market in MARKETS:
        df_m = df_drugs[df_drugs["Website"] == market].copy()
        G = get_network(market, df_drugs)
        if G is None or G.number_of_nodes() == 0:
            print(f"[skip] {market}: no graph")
            continue
        cent = centrality_frame(market, G, prod_to_cat2, count_reviews_by_product(df_m))
        supply = product_vendor_supply_stats(df_m, prod_to_cat2)
        merged = cent.merge(supply, on="product_cat3", how="left")
        frames.append(merged)
        print(f"{market}: {len(merged)} products in panel from graph n={G.number_of_nodes()}, m={G.number_of_edges()}")

    if not frames:
        raise RuntimeError("No panel rows. Check selected network files/settings.")

    panel = pd.concat(frames, ignore_index=True)
    cat2_size = category2_distinct_cat3_per_market(df_drugs, prod_to_cat2)
    panel = panel.merge(cat2_size, on=["market", "category2"], how="left")
    panel["n_cat3_in_cat2"] = panel["n_cat3_in_cat2"].fillna(0).astype(int)

    log_cols = [
        "n_cat3_in_cat2",
        "n_sellers",
        "mean_L",
        "max_L",
        "mean_K",
        "max_K",
        "mean_cat2_scope",
        "max_cat2_scope",
        "product_total_listings",
        "mean_vendor_total_listings",
        "max_vendor_total_listings",
    ]
    for col in log_cols:
        panel[f"log1p_{col}"] = np.log1p(panel[col].fillna(0))

    needed = [
        "degree_centrality",
        "weighted_degree",
        "log1p_reviews",
        "log1p_n_cat3_in_cat2",
        "log1p_n_sellers",
        "log1p_mean_L",
        "log1p_max_L",
        "log1p_mean_cat2_scope",
        "log1p_max_cat2_scope",
        "log1p_product_total_listings",
    ]
    panel = panel.dropna(subset=needed).copy()
    return panel

panel = build_panel()
panel_path = TAB / f"{PREFIX}_product_panel.csv"
panel.to_csv(panel_path, index=False)
print("Saved panel:", panel_path)
display(panel.head())
panel.groupby("market").size().rename("n_products").to_frame()




In [ ]:
panel.columns

## Run the regressions



In [ ]:
def require_reference_level(column: str, reference: str) -> str:
    levels = set(panel[column].astype(str))
    if reference not in levels:
        available = ", ".join(sorted(levels))
        raise ValueError(f"Reference level '{reference}' is not present in {column}. Available levels: {available}")
    return reference


def market_fe() -> str:
    ref = require_reference_level("market", MARKET_REF)
    return f"C(market, Treatment(reference='{ref}'))"


def category2_fe() -> str:
    ref = require_reference_level("category2", CATEGORY2_REF)
    return f"C(category2, Treatment(reference='{ref}'))"

STRUCTURE_TERMS = " + ".join([
    "log1p_reviews",
    "log1p_n_cat3_in_cat2",
    "log1p_n_sellers",
    "log1p_mean_L",
    "log1p_max_L",
    "log1p_mean_cat2_scope",
    "log1p_max_cat2_scope",
    "log1p_product_total_listings",
])

BASE_RHS = f"{STRUCTURE_TERMS} + {market_fe()}"
CAT_RHS = f"{BASE_RHS} + {category2_fe()}"

MODEL_SPECS = [
    {"model": "degree_structure_market", "outcome": "degree_centrality", "rhs": BASE_RHS},
    {"model": "degree_structure_market_rx", "outcome": "degree_centrality", "rhs": f"{BASE_RHS} + is_prescription"},
    {"model": "degree_structure_market_category2", "outcome": "degree_centrality", "rhs": CAT_RHS},
    {"model": "wdegree_structure_market", "outcome": "weighted_degree", "rhs": BASE_RHS},
    {"model": "wdegree_structure_market_rx", "outcome": "weighted_degree", "rhs": f"{BASE_RHS} + is_prescription"},
    {"model": "wdegree_structure_market_category2", "outcome": "weighted_degree", "rhs": CAT_RHS},
]

models = {}
summary_rows = []
coef_rows = []

print("Running OLS regressions with statsmodels.formula.api.ols")
print("Market reference category:", MARKET_REF)
print("Category 2 reference category:", CATEGORY2_REF)
print("Model formulas:")
for spec in MODEL_SPECS:
    print(f"- {spec['model']}: {spec['outcome']} ~ {spec['rhs']}")

for spec in MODEL_SPECS:
    formula = f"{spec['outcome']} ~ {spec['rhs']}"
    fit_kwargs = {} if COVARIANCE_TYPE == "nonrobust" else {"cov_type": COVARIANCE_TYPE}
    res = smf.ols(formula, data=panel).fit(
            cov_type="cluster", cov_kwds={"groups": panel["product_cat3"]}
        )
    models[spec["model"]] = res
    summary_rows.append({
        "network_label": network_label(),
        "model": spec["model"],
        "outcome": spec["outcome"],
        "formula": formula,
        "nobs": int(res.nobs),
        "r2": float(res.rsquared),
        "adj_r2": float(res.rsquared_adj),
        "aic": float(res.aic),
        "bic": float(res.bic),
        "covariance": COVARIANCE_TYPE,
    })
    ci = res.conf_int()
    for term in res.params.index:
        coef_rows.append({
            "network_label": network_label(),
            "model": spec["model"],
            "outcome": spec["outcome"],
            "term": term,
            "coef": float(res.params[term]),
            "std_err": float(res.bse[term]),
            "t": float(res.tvalues[term]),
            "p_value": float(res.pvalues[term]),
            "ci_low": float(ci.loc[term, 0]),
            "ci_high": float(ci.loc[term, 1]),
        })

model_summary = pd.DataFrame(summary_rows)
coefficients = pd.DataFrame(coef_rows)
summary_path = TAB / f"{PREFIX}_model_summary.csv"
coef_path = TAB / f"{PREFIX}_coefficients.csv"
model_summary.to_csv(summary_path, index=False)
coefficients.to_csv(coef_path, index=False)
print("Saved model summary:", summary_path)
print("Saved coefficients:", coef_path)
display(model_summary[["model", "outcome", "nobs", "r2", "adj_r2", "covariance"]])





In [ ]:
model_summary

## Nested prescription-coefficient decomposition

Sequentially adds structural and vendor-portfolio controls to assess changes in the prescription coefficient.


In [ ]:
NESTED_CONTROL_STEPS = [
    {
        "step": 1,
        "model_stage": "Reviews + market FE",
        "controls_added": "Demand proxy and platform differences",
        "rhs_terms": ["log1p_reviews", market_fe(), "is_prescription"],
    },
    {
        "step": 2,
        "model_stage": "+ sellers",
        "controls_added": "Number of distinct sellers for the product",
        "rhs_terms": ["log1p_reviews", "log1p_n_sellers", market_fe(), "is_prescription"],
    },
    {
        "step": 3,
        "model_stage": "+ product listings",
        "controls_added": "Product-level listing supply",
        "rhs_terms": ["log1p_reviews", "log1p_n_sellers", "log1p_product_total_listings", market_fe(), "is_prescription"],
    },
    {
        "step": 4,
        "model_stage": "+ vendor Category 2 scope",
        "controls_added": "Mean and maximum cross-category scope of supplying vendors",
        "rhs_terms": [
            "log1p_reviews",
            "log1p_n_sellers",
            "log1p_product_total_listings",
            "log1p_mean_cat2_scope",
            "log1p_max_cat2_scope",
            market_fe(),
            "is_prescription",
        ],
    },
    {
        "step": 5,
        "model_stage": "+ Category 2 product count",
        "controls_added": "Number of Category 3 products in the broad category",
        "rhs_terms": [
            "log1p_reviews",
            "log1p_n_sellers",
            "log1p_product_total_listings",
            "log1p_mean_cat2_scope",
            "log1p_max_cat2_scope",
            "log1p_n_cat3_in_cat2",
            market_fe(),
            "is_prescription",
        ],
    },
    {
        "step": 6,
        "model_stage": "Full model",
        "controls_added": "Adds vendor listing scale",
        "rhs_terms": [
            "log1p_reviews",
            "log1p_n_sellers",
            "log1p_product_total_listings",
            "log1p_mean_cat2_scope",
            "log1p_max_cat2_scope",
            "log1p_n_cat3_in_cat2",
            "log1p_mean_L",
            "log1p_max_L",
            market_fe(),
            "is_prescription",
        ],
    },
]

NESTED_OUTCOMES = [
    ("degree_centrality", "Degree centrality"),
    ("weighted_degree", "Weighted degree"),
]

nested_rows = []
nested_model_rows = []
for outcome, outcome_label in NESTED_OUTCOMES:
    for step in NESTED_CONTROL_STEPS:
        rhs = " + ".join(step["rhs_terms"])
        formula = f"{outcome} ~ {rhs}"
        res = smf.ols(formula, data=panel).fit(
            cov_type="cluster", cov_kwds={"groups": panel["product_cat3"]}
        )
        ci = res.conf_int()
        term = "is_prescription"
        nested_rows.append({
            "network_label": network_label(),
            "outcome": outcome,
            "outcome_label": outcome_label,
            "step": step["step"],
            "model_stage": step["model_stage"],
            "controls_added": step["controls_added"],
            "term": term,
            "coef": float(res.params[term]),
            "std_err": float(res.bse[term]),
            "t": float(res.tvalues[term]),
            "p_value": float(res.pvalues[term]),
            "ci_low": float(ci.loc[term, 0]),
            "ci_high": float(ci.loc[term, 1]),
            "nobs": int(res.nobs),
            "r2": float(res.rsquared),
            "adj_r2": float(res.rsquared_adj),
            "covariance": COVARIANCE_TYPE,
            "formula": formula,
        })
        nested_model_rows.append({
            "network_label": network_label(),
            "outcome": outcome,
            "step": step["step"],
            "model_stage": step["model_stage"],
            "controls_added": step["controls_added"],
            "formula": formula,
            "nobs": int(res.nobs),
            "r2": float(res.rsquared),
            "adj_r2": float(res.rsquared_adj),
            "aic": float(res.aic),
            "bic": float(res.bic),
            "covariance": COVARIANCE_TYPE,
        })

nested_rx = pd.DataFrame(nested_rows)
nested_models = pd.DataFrame(nested_model_rows)

nested_rx["stars"] = nested_rx["p_value"].map(stars)
nested_rx["coef_se"] = nested_rx.apply(lambda r: f"{r['coef']:.4f}{r['stars']} ({r['std_err']:.4f})", axis=1)
nested_wide = nested_rx.pivot(index=["step", "model_stage", "controls_added"], columns="outcome_label", values="coef_se").reset_index()

nested_rx_path = TAB / f"{PREFIX}_nested_prescription_coefficient_decomposition.csv"
nested_model_path = TAB / f"{PREFIX}_nested_model_summary.csv"
nested_wide_path = TAB / f"{PREFIX}_nested_prescription_coefficient_decomposition_readable.csv"
nested_rx.to_csv(nested_rx_path, index=False)
nested_models.to_csv(nested_model_path, index=False)
nested_wide.to_csv(nested_wide_path, index=False)

print("Saved nested prescription-coefficient decomposition:", nested_rx_path)
print("Saved nested model summary:", nested_model_path)
print("Saved readable nested table:", nested_wide_path)
display(nested_wide)
